# Influencer Fraud / Fake-Engagement Detection System

**Client:** Marketing Analytics Agency
**Purpose:** Score Instagram/TikTok handles 0-100 for fake-engagement risk before campaign approval, using
unsupervised anomaly detection (Isolation Forest + Local Outlier Factor) on public profile metrics.

**Pipeline**
1. Collect/simulate 5,000 influencer profiles (public metrics only: followers, following, posts, likes, comments, growth rate, etc.)
2. Engineer engagement-ratio and behavioral features
3. Manually label a 200-profile validation sample (fraud / genuine) using domain heuristics, treated as ground truth
4. Train Isolation Forest & Local Outlier Factor (unsupervised, unlabeled data)
5. Combine into a single 0-100 Fraud Risk Score, tune the decision threshold against the labeled sample
6. Validate against a held-out "known fake influencer" stress-test set
7. Map score bands to concrete vetting actions and export deliverables

> **Data note:** SocialBlade does not offer a public self-serve API suitable for bulk automated scoring, and
> scraping Instagram/TikTok directly violates their Terms of Service. This notebook therefore uses a carefully
> constructed **synthetic dataset that reproduces documented statistical signatures of real fake-engagement
> schemes** (follower/engagement mismatches, bot-farm growth spikes, comment-pod pattern, engagement-rate
> distributions reported in industry fraud studies). The code is written so that swapping in a real data source
> (SocialBlade export, HypeAuditor export, or an internal scraper) only requires replacing the `generate_profiles()`
> function — every downstream step (features, models, scoring, thresholds) is unchanged.

In [21]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Data Collection

We simulate 5,000 public profiles across Instagram and TikTok. Roughly 12% are generated as genuinely
fraudulent/fake-engagement accounts (bot-follower purchases, engagement pods, growth-spike bot farms,
comment-spam pods) — a proportion consistent with published influencer-fraud prevalence estimates
(industry studies typically report 10-15% of accounts showing significant fake-engagement signals).
The other ~88% are simulated as authentic accounts with natural, noisy-but-consistent engagement behavior.

Every field below is a metric that is publicly visible on a profile or derivable from a handful of recent
posts (no private data), matching what an agency could pull from SocialBlade, HypeAuditor, or manual review.

In [22]:
def generate_profiles(n=5000, fraud_rate=0.12, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n_fraud = int(n * fraud_rate)
    n_genuine = n - n_fraud
    rows = []

    # ---- Genuine accounts -------------------------------------------------
    # Followers log-normally distributed (mimics real creator-size distribution, nano -> mega influencer)
    followers = rng.lognormal(mean=9.2, sigma=1.6, size=n_genuine).clip(500, 20_000_000)
    # Real engagement rate shrinks as follower count grows (well-documented "reach decay") + noise
    base_er = 0.11 * (followers / 1000) ** -0.18
    engagement_rate = np.clip(base_er * rng.normal(1.0, 0.28, n_genuine), 0.003, 0.35)
    avg_likes = followers * engagement_rate * rng.uniform(0.75, 0.9, n_genuine)
    avg_comments = avg_likes * rng.uniform(0.02, 0.09, n_genuine)
    following = rng.lognormal(mean=6.0, sigma=1.0, size=n_genuine).clip(20, 50_000)
    posts = rng.lognormal(mean=5.3, sigma=1.0, size=n_genuine).clip(3, 15_000)
    account_age_days = rng.uniform(180, 4000, n_genuine)
    follower_growth_30d_pct = rng.normal(2.0, 4.0, n_genuine).clip(-15, 40)
    comment_like_ratio = avg_comments / np.maximum(avg_likes, 1)
    like_to_follower_std = rng.uniform(0.15, 0.45, n_genuine)  # variability of per-post like rate
    pct_comments_generic = rng.beta(2, 10, n_genuine)          # share of comments that are generic ("nice!", emojis)
    posting_regularity_cv = rng.uniform(0.2, 0.8, n_genuine)   # coefficient of variation of posting gaps
    audience_overlap_score = rng.uniform(0.0, 0.25, n_genuine) # overlap with known bot-farm follower clusters
    label = np.zeros(n_genuine, dtype=int)
    platform = rng.choice(["Instagram", "TikTok"], n_genuine, p=[0.6, 0.4])

    genuine = pd.DataFrame({
        "followers": followers, "following": following, "posts": posts,
        "avg_likes": avg_likes, "avg_comments": avg_comments,
        "account_age_days": account_age_days,
        "follower_growth_30d_pct": follower_growth_30d_pct,
        "comment_like_ratio": comment_like_ratio,
        "like_to_follower_std": like_to_follower_std,
        "pct_comments_generic": pct_comments_generic,
        "posting_regularity_cv": posting_regularity_cv,
        "audience_overlap_score": audience_overlap_score,
        "platform": platform, "label": label,
    })

    # ---- Fraudulent accounts (mixed fraud archetypes) ---------------------
    sub_n = n_fraud // 3
    remainder = n_fraud - sub_n * 3

    # Archetype A: Purchased bot followers -> big follower count, engagement rate collapses
    followers_a = rng.lognormal(mean=10.5, sigma=1.3, size=sub_n).clip(2000, 15_000_000)
    engagement_rate_a = np.clip(rng.normal(0.006, 0.004, sub_n), 0.0003, 0.02)
    avg_likes_a = followers_a * engagement_rate_a
    avg_comments_a = avg_likes_a * rng.uniform(0.001, 0.01, sub_n)  # comments collapse even more than likes

    # Archetype B: Engagement-pod / comment-spam -> engagement rate looks artificially HIGH & generic
    followers_b = rng.lognormal(mean=8.6, sigma=1.4, size=sub_n).clip(1000, 3_000_000)
    engagement_rate_b = np.clip(rng.normal(0.22, 0.09, sub_n), 0.08, 0.55)  # unnaturally high, uniform
    avg_likes_b = followers_b * engagement_rate_b
    avg_comments_b = avg_likes_b * rng.uniform(0.12, 0.30, sub_n)  # heavy generic comment pods

    # Archetype C: Bot-farm growth spike -> sudden follower spike, stale low engagement
    followers_c = rng.lognormal(mean=9.8, sigma=1.5, size=sub_n + remainder).clip(1500, 8_000_000)
    engagement_rate_c = np.clip(rng.normal(0.015, 0.01, sub_n + remainder), 0.001, 0.06)
    avg_likes_c = followers_c * engagement_rate_c
    avg_comments_c = avg_likes_c * rng.uniform(0.005, 0.03, sub_n + remainder)

    def pack(followers_x, avg_likes_x, avg_comments_x, k, growth, generic, overlap, reg_cv, like_std):
        following_x = rng.lognormal(mean=6.5, sigma=1.3, size=k).clip(20, 80_000)
        posts_x = rng.lognormal(mean=4.6, sigma=1.2, size=k).clip(1, 8000)
        age_x = rng.uniform(30, 2500, k)
        return pd.DataFrame({
            "followers": followers_x, "following": following_x, "posts": posts_x,
            "avg_likes": avg_likes_x, "avg_comments": avg_comments_x,
            "account_age_days": age_x,
            "follower_growth_30d_pct": growth,
            "comment_like_ratio": avg_comments_x / np.maximum(avg_likes_x, 1),
            "like_to_follower_std": like_std,
            "pct_comments_generic": generic,
            "posting_regularity_cv": reg_cv,
            "audience_overlap_score": overlap,
        })

    fraud_a = pack(followers_a, avg_likes_a, avg_comments_a, sub_n,
                    growth=rng.normal(45, 25, sub_n).clip(5, 200),          # spiky purchase-driven growth
                    generic=rng.beta(2, 6, sub_n),
                    overlap=rng.uniform(0.35, 0.85, sub_n),
                    reg_cv=rng.uniform(0.6, 1.6, sub_n),
                    like_std=rng.uniform(0.5, 1.4, sub_n))
    fraud_b = pack(followers_b, avg_likes_b, avg_comments_b, sub_n,
                    growth=rng.normal(8, 6, sub_n).clip(-5, 40),
                    generic=rng.beta(8, 2, sub_n),                          # mostly generic "pod" comments
                    overlap=rng.uniform(0.30, 0.75, sub_n),
                    reg_cv=rng.uniform(0.05, 0.25, sub_n),                  # unnaturally REGULAR posting (scheduled pods)
                    like_std=rng.uniform(0.02, 0.12, sub_n))                # unnaturally CONSTANT like rate
    fraud_c = pack(followers_c, avg_likes_c, avg_comments_c, sub_n + remainder,
                    growth=rng.normal(120, 60, sub_n + remainder).clip(30, 500),  # explosive spike
                    generic=rng.beta(2, 5, sub_n + remainder),
                    overlap=rng.uniform(0.4, 0.9, sub_n + remainder),
                    reg_cv=rng.uniform(0.5, 1.5, sub_n + remainder),
                    like_std=rng.uniform(0.4, 1.2, sub_n + remainder))

    fraud = pd.concat([fraud_a, fraud_b, fraud_c], ignore_index=True)
    fraud["platform"] = rng.choice(["Instagram", "TikTok"], len(fraud), p=[0.55, 0.45])
    fraud["label"] = 1

    df = pd.concat([genuine, fraud], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df.insert(0, "handle", ["@influencer_" + str(i).zfill(5) for i in range(len(df))])
    for c in ["followers", "following", "posts", "avg_likes", "avg_comments"]:
        df[c] = df[c].round(0).astype(int)
    return df

profiles = generate_profiles(n=5000, fraud_rate=0.12)
print(f"Generated {len(profiles)} profiles | fraud (ground-truth, hidden from models): {profiles['label'].mean():.1%}")
profiles.drop(columns=["label"]).head(15)

Generated 5000 profiles | fraud (ground-truth, hidden from models): 12.0%


,handle,followers,following,posts,avg_likes,avg_comments,account_age_days,follower_growth_30d_pct,comment_like_ratio,like_to_follower_std,pct_comments_generic,posting_regularity_cv,audience_overlap_score,platform
0,@influencer_00000,3643,287,298,250,12,374.876408,1.880556,0.048496,0.434321,0.064356,0.625835,0.197937,Instagram
1,@influencer_00001,11298,748,168,446,39,888.786045,5.636692,0.086954,0.344106,0.244504,0.405779,0.017076,Instagram
2,@influencer_00002,2636,595,226,305,14,3190.159802,7.372262,0.045894,0.217655,0.056824,0.204026,0.141155,Instagram
3,@influencer_00003,104321,186,261,5123,341,1669.151568,4.649826,0.066482,0.189393,0.039627,0.219576,0.196786,TikTok
4,@influencer_00004,2077,822,142,127,3,1965.333148,6.946471,0.024411,0.157868,0.190990,0.353745,0.151651,TikTok
5,@influencer_00005,2165,2095,71,194,8,2759.888729,-3.331894,0.043811,0.411576,0.351186,0.292366,0.049990,TikTok
6,@influencer_00006,130325,403,36,6108,322,761.815317,-3.027321,0.052651,0.217966,0.046496,0.264258,0.131196,Instagram
7,@influencer_00007,28542,737,16,890,58,2641.549194,-4.505268,0.064929,0.418837,0.256630,0.531471,0.135946,Instagram
8,@influencer_00008,986,291,69,58,4,933.570978,-0.291229,0.062451,0.201163,0.178598,0.349466,0.240203,TikTok
9,@influencer_00009,9064,1872,594,556,17,725.494737,-0.897584,0.030715,0.236856,0.050156,0.372036,0.128743,TikTok


In [23]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   handle                   5000 non-null   object 
 1   followers                5000 non-null   int64  
 2   following                5000 non-null   int64  
 3   posts                    5000 non-null   int64  
 4   avg_likes                5000 non-null   int64  
 5   avg_comments             5000 non-null   int64  
 6   account_age_days         5000 non-null   float64
 7   follower_growth_30d_pct  5000 non-null   float64
 8   comment_like_ratio       5000 non-null   float64
 9   like_to_follower_std     5000 non-null   float64
 10  pct_comments_generic     5000 non-null   float64
 11  posting_regularity_cv    5000 non-null   float64
 12  audience_overlap_score   5000 non-null   float64
 13  platform                 5000 non-null   object 
 14  label                   

In [24]:
profiles.describe().T

,count,mean,std,min,25%,50%,75%,max
followers,5000.0,36373.739200,101787.238972,500.000000,3416.500000,10389.000000,29228.250000,2.486718e+06
following,5000.0,823.695600,1599.283330,20.000000,215.000000,444.000000,902.250000,6.365700e+04
posts,5000.0,320.294600,413.010636,2.000000,96.000000,194.000000,373.000000,5.583000e+03
avg_likes,5000.0,1398.222600,3085.269032,1.000000,207.750000,547.000000,1348.250000,7.556500e+04
avg_comments,5000.0,91.660000,265.937736,0.000000,9.000000,27.000000,74.000000,6.797000e+03
account_age_days,5000.0,1998.463861,1091.908612,36.058555,1066.170068,1965.140506,2908.540122,3.999836e+03
follower_growth_30d_pct,5000.0,8.863871,28.242129,-15.000000,-0.316112,2.530118,5.917806,2.994472e+02
comment_like_ratio,5000.0,0.058335,0.040357,0.001004,0.033916,0.054301,0.074198,2.997442e-01
like_to_follower_std,5000.0,0.340466,0.202423,0.020345,0.223070,0.309988,0.395093,1.393887e+00
pct_comments_generic,5000.0,0.198803,0.163016,0.000958,0.092638,0.159507,0.250692,9.873811e-01


In [25]:
profiles.isnull().sum()

handle                     0
followers                  0
following                  0
posts                      0
avg_likes                  0
avg_comments               0
account_age_days           0
follower_growth_30d_pct    0
comment_like_ratio         0
like_to_follower_std       0
pct_comments_generic       0
posting_regularity_cv      0
audience_overlap_score     0
platform                   0
label                      0
dtype: int64

## 2. Feature Engineering

We derive the ratio-based, distribution-shape, and behavioral-regularity features that the fraud-detection
literature and platforms like HypeAuditor/SocialBlade use as fake-engagement signals:

| Feature | Fraud signal |
|---|---|
| `engagement_rate` | Purchased-follower accounts collapse; pod accounts spike unnaturally high |
| `comment_like_ratio` | Very low (bought likes, no real comments) or very high + generic (comment pods) |
| `follower_following_ratio` | Bot/pod accounts often follow-for-follow, deflating this ratio |
| `follower_growth_30d_pct` | Organic growth is slow & noisy; bot purchases create spikes |
| `posts_per_day` | Sanity-checks activity level vs. account age |
| `like_to_follower_std` (z) | Real engagement is noisy post-to-post; pods are unnaturally constant |
| `pct_comments_generic` | Share of low-effort/generic comments (emoji-only, "nice pic!") |
| `posting_regularity_cv` | Extremely regular (bot-scheduled) or extremely erratic posting |
| `audience_overlap_score` | Overlap with known bot-farm/pod follower clusters |

In [26]:
def engineer_features(df):
    out = df.copy()
    out["engagement_rate"] = (out["avg_likes"] + out["avg_comments"]) / out["followers"].clip(lower=1)
    out["comment_like_ratio"] = out["comment_like_ratio"].clip(upper=5)
    out["follower_following_ratio"] = out["followers"] / out["following"].clip(lower=1)
    out["posts_per_day"] = out["posts"] / (out["account_age_days"].clip(lower=1))
    out["log_followers"] = np.log1p(out["followers"])
    # Engagement-rate residual vs. the expected reach-decay curve for that follower count
    expected_er = 0.11 * (out["followers"].clip(lower=500) / 1000) ** -0.18
    out["engagement_rate_deviation"] = (out["engagement_rate"] - expected_er) / expected_er
    return out

feat_df = engineer_features(profiles)

FEATURES = [
    "log_followers", "follower_following_ratio", "engagement_rate",
    "comment_like_ratio", "follower_growth_30d_pct", "posts_per_day",
    "like_to_follower_std", "pct_comments_generic", "posting_regularity_cv",
    "audience_overlap_score", "engagement_rate_deviation",
]
feat_df[FEATURES].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
log_followers,9.250070,1.558960,6.216606,14.726475
follower_following_ratio,136.902751,525.102744,0.035376,11403.318966
engagement_rate,0.069401,0.052652,0.000283,0.569361
comment_like_ratio,0.058335,0.040357,0.001004,0.299744
follower_growth_30d_pct,8.863871,28.242129,-15.000000,299.447188
posts_per_day,0.279400,0.631487,0.001558,14.913031
like_to_follower_std,0.340466,0.202423,0.020345,1.393887
pct_comments_generic,0.198803,0.163016,0.000958,0.987381
posting_regularity_cv,0.531398,0.243256,0.050097,1.595750
audience_overlap_score,0.182443,0.173797,0.000003,0.890263


## 3. Manual Labeling of the 200-Profile Validation Sample

Per the project brief, a **200-profile stratified sample** is manually labeled by a reviewer (here, sampled
from our ground-truth generator to stand in for human review) and held out purely for **tuning the decision
threshold and reporting precision/recall** — it is never shown to the unsupervised models during fitting,
which train on the full unlabeled 5,000-profile pool as intended.

In [27]:
def stratified_label_sample(df, n=200, seed=RANDOM_STATE):
    rng_local = np.random.default_rng(seed)
    fraud_idx = df.index[df["label"] == 1]
    genuine_idx = df.index[df["label"] == 0]
    # Oversample the fraud class relative to its 12% base rate so the labeled set has enough
    # positive examples to measure precision reliably (a standard practice for rare-class validation)
    n_fraud_labeled = int(n * 0.35)
    n_genuine_labeled = n - n_fraud_labeled
    sampled_fraud = rng_local.choice(fraud_idx, size=n_fraud_labeled, replace=False)
    sampled_genuine = rng_local.choice(genuine_idx, size=n_genuine_labeled, replace=False)
    labeled_idx = np.concatenate([sampled_fraud, sampled_genuine])
    rng_local.shuffle(labeled_idx)
    return labeled_idx

labeled_idx = stratified_label_sample(feat_df, n=200)
feat_df["manually_labeled"] = False
feat_df.loc[labeled_idx, "manually_labeled"] = True
print(f"Labeled sample: {len(labeled_idx)} profiles | "
      f"fraud in sample: {feat_df.loc[labeled_idx, 'label'].sum()} "
      f"({feat_df.loc[labeled_idx, 'label'].mean():.1%})")

Labeled sample: 200 profiles | fraud in sample: 70 (35.0%)


## 4. Unsupervised Models: Isolation Forest & Local Outlier Factor

Both models are fit on the **full 5,000-profile pool, completely unsupervised** (no labels used in fitting) —
only the held-out 200-profile labeled sample is used afterward, purely to tune the score threshold and report
precision/recall. Features are robust-scaled (median/IQR) first since engagement metrics are heavily skewed
and we don't want a handful of mega-influencers to dominate distances/splits.

In [28]:
X = feat_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(
    n_estimators=300, contamination=0.12, random_state=RANDOM_STATE, n_jobs=-1
)
iso_forest.fit(X_scaled)
# score_samples: higher = more normal. Flip sign so higher = more anomalous, then min-max to [0,1]
iso_raw = -iso_forest.score_samples(X_scaled)
iso_norm = (iso_raw - iso_raw.min()) / (iso_raw.max() - iso_raw.min())

lof = LocalOutlierFactor(n_neighbors=35, contamination=0.12, novelty=False, n_jobs=-1)
lof_pred = lof.fit_predict(X_scaled)          # -1 outlier / 1 inlier, only available for the fitted set
lof_raw = -lof.negative_outlier_factor_       # higher = more anomalous
lof_norm = (lof_raw - lof_raw.min()) / (lof_raw.max() - lof_raw.min())

feat_df["iso_forest_score"] = iso_norm
feat_df["lof_score"] = lof_norm
print("Isolation Forest score range:", iso_norm.min().round(3), "-", iso_norm.max().round(3))
print("LOF score range:", lof_norm.min().round(3), "-", lof_norm.max().round(3))
print("\nCorrelation between the two unsupervised scores:", np.corrcoef(iso_norm, lof_norm)[0, 1].round(3))

Isolation Forest score range: 0.0 - 1.0
LOF score range: 0.0 - 1.0

Correlation between the two unsupervised scores: 0.357


## 5. Ensemble Fraud Risk Score (0-100)

The two models catch different fraud archetypes (Isolation Forest is better at global, "off in every
dimension" outliers like bot-farm spikes; LOF is better at local-density anomalies like engagement pods that
only look strange relative to *similar-sized* accounts). We average them 50/50 and rescale to a 0-100 score
that a non-technical vetting team can act on directly.

In [39]:
feat_df["fraud_risk_score"] = ((0.5 * feat_df["iso_forest_score"] + 0.5 * feat_df["lof_score"]) * 100).round(1)

plt.figure(figsize=(7, 4))
plt.hist(feat_df.loc[feat_df.label == 0, "fraud_risk_score"], bins=40, alpha=0.6, label="Genuine (ground truth)")
plt.hist(feat_df.loc[feat_df.label == 1, "fraud_risk_score"], bins=40, alpha=0.6, label="Fraud (ground truth)")
plt.xlabel("Fraud Risk Score (0-100)"); plt.ylabel("Count")
plt.title("Fraud Risk Score distribution by ground-truth class")
plt.legend(); plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/score_distribution.png", dpi=140)
plt.show()

/var/folders/12/c0j5dnhn29qccm23n3101t880000gn/T/ipykernel_5838/3567070434.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Threshold Tuning & Validation (≥85% precision target)

Using only the 200-profile manually labeled sample, we sweep the score threshold and pick the lowest
threshold that still clears **85% precision on the "fraud" class** — lowest, so we flag as many true fraud
cases as the precision budget allows (maximizing recall subject to the precision constraint), which is the
right trade-off for a pre-campaign vetting gate.

In [41]:
val = feat_df.loc[feat_df["manually_labeled"]].copy()
y_true = val["label"].values
scores = val["fraud_risk_score"].values

thresholds = np.arange(5, 96, 1)
rows = []
for t in thresholds:
    y_pred = (scores >= t).astype(int)
    if y_pred.sum() == 0:
        continue
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    rows.append((t, p, r, f1, y_pred.sum()))
sweep = pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1", "n_flagged"])

qualifying = sweep[sweep["precision"] >= 0.85]
CHOSEN_THRESHOLD = int(qualifying["threshold"].min()) if len(qualifying) else int(sweep.loc[sweep["precision"].idxmax(), "threshold"])
chosen_row = sweep.loc[sweep["threshold"] == CHOSEN_THRESHOLD].iloc[0]
print(f"Chosen HIGH-RISK threshold: {CHOSEN_THRESHOLD}")
print(chosen_row)

plt.figure(figsize=(7, 4))
plt.plot(sweep["threshold"], sweep["precision"], label="Precision")
plt.plot(sweep["threshold"], sweep["recall"], label="Recall")
plt.axhline(0.85, color="gray", linestyle="--", linewidth=1, label="85% precision target")
plt.axvline(CHOSEN_THRESHOLD, color="red", linestyle=":", linewidth=1, label=f"Chosen threshold = {CHOSEN_THRESHOLD}")
plt.xlabel("Fraud Risk Score threshold"); plt.ylabel("Score")
plt.title("Precision / Recall vs. threshold (200-profile labeled sample)")
plt.legend(); plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/precision_recall_sweep.png", dpi=140)
plt.show()

y_pred_final = (scores >= CHOSEN_THRESHOLD).astype(int)
final_precision = precision_score(y_true, y_pred_final, zero_division=0)
final_recall = recall_score(y_true, y_pred_final, zero_division=0)
final_f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final)
print(f"\nFINAL VALIDATION @ threshold={CHOSEN_THRESHOLD}")
print(f"Precision: {final_precision:.3f}  (target >= 0.85 -> {'PASS' if final_precision >= 0.85 else 'FAIL'})")
print(f"Recall:    {final_recall:.3f}")
print(f"F1:        {final_f1:.3f}")
print("Confusion matrix [rows=true(0,1), cols=pred(0,1)]:\n", cm)

Chosen HIGH-RISK threshold: 16
threshold    16.000000
precision     0.864198
recall        1.000000
f1            0.927152
n_flagged    81.000000
Name: 11, dtype: float64

FINAL VALIDATION @ threshold=16
Precision: 0.864  (target >= 0.85 -> PASS)
Recall:    1.000
F1:        0.927
Confusion matrix [rows=true(0,1), cols=pred(0,1)]:
 [[119  11]
 [  0  70]]


/var/folders/12/c0j5dnhn29qccm23n3101t880000gn/T/ipykernel_5838/151869745.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Vetting Action Bands

The chosen high-risk threshold anchors a 4-band action policy the agency's vetting team can apply directly.
Bands below the high-risk threshold are spaced to give a graded "watchlist" zone rather than a hard cliff.

In [43]:
def score_to_action(score, high_risk_threshold=CHOSEN_THRESHOLD):
    watch_low = max(high_risk_threshold - 30, 10)
    watch_high = high_risk_threshold
    caution_high = min(high_risk_threshold + 15, 100)
    if score < watch_low:
        return "Approve"
    elif score < watch_high:
        return "Manual Review — Low Concern"
    elif score < caution_high:
        return "Manual Review — High Concern"
    else:
        return "Reject / Escalate"

feat_df["vetting_action"] = feat_df["fraud_risk_score"].apply(score_to_action)
feat_df["vetting_action"].value_counts()

vetting_action
Approve                         3669
Manual Review — Low Concern      553
Manual Review — High Concern     446
Reject / Escalate                332
Name: count, dtype: int64

## 8. Stress-Test: Validation Against "Known Fake Influencer" Patterns

As required by the brief, we validate against a **held-out set of known/extreme fake-influencer profiles**
that the models never saw during fitting or threshold tuning — engineered to match documented fraud case
studies (e.g. mega-follower accounts with near-zero engagement; accounts with 1,000%+ month-over-month
follower spikes typical of bulk bot purchases).

In [44]:
def generate_known_fake_holdout(n=150, seed=999):
    rng_h = np.random.default_rng(seed)
    followers = rng_h.lognormal(mean=11.0, sigma=1.0, size=n).clip(5000, 20_000_000)
    engagement_rate = np.clip(rng_h.normal(0.004, 0.003, n), 0.0001, 0.015)
    avg_likes = followers * engagement_rate
    avg_comments = avg_likes * rng_h.uniform(0.001, 0.008, n)
    following = rng_h.lognormal(mean=6.0, sigma=1.2, size=n).clip(20, 60_000)
    posts = rng_h.lognormal(mean=4.8, sigma=1.1, size=n).clip(1, 5000)
    age = rng_h.uniform(60, 1800, n)
    growth = rng_h.normal(180, 90, n).clip(50, 900)   # extreme documented bot-purchase spikes
    df = pd.DataFrame({
        "followers": followers.round(0).astype(int), "following": following.round(0).astype(int),
        "posts": posts.round(0).astype(int), "avg_likes": avg_likes.round(0).astype(int),
        "avg_comments": avg_comments.round(0).astype(int), "account_age_days": age,
        "follower_growth_30d_pct": growth,
        "comment_like_ratio": avg_comments / np.maximum(avg_likes, 1),
        "like_to_follower_std": rng_h.uniform(0.6, 1.5, n),
        "pct_comments_generic": rng_h.beta(2, 5, n),
        "posting_regularity_cv": rng_h.uniform(0.6, 1.6, n),
        "audience_overlap_score": rng_h.uniform(0.5, 0.95, n),
        "platform": rng_h.choice(["Instagram", "TikTok"], n),
        "label": 1,
    })
    df.insert(0, "handle", ["@knownfake_" + str(i).zfill(4) for i in range(n)])
    return df

holdout_raw = generate_known_fake_holdout(150)
holdout_feat = engineer_features(holdout_raw)
X_holdout = scaler.transform(holdout_feat[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0))

iso_h_raw = -iso_forest.score_samples(X_holdout)
iso_h_norm = (iso_h_raw - iso_raw.min()) / (iso_raw.max() - iso_raw.min())
iso_h_norm = iso_h_norm.clip(0, 1)

# LOF has no separate .decision_function without novelty=True; refit a novelty-mode LOF on the training
# distribution for scoring genuinely new/unseen points such as this holdout set
lof_novel = LocalOutlierFactor(n_neighbors=35, contamination=0.12, novelty=True, n_jobs=-1)
lof_novel.fit(X_scaled)
lof_h_raw = -lof_novel.score_samples(X_holdout)
lof_h_norm = ((lof_h_raw - lof_raw.min()) / (lof_raw.max() - lof_raw.min())).clip(0, 1)

holdout_feat["fraud_risk_score"] = ((0.5 * iso_h_norm + 0.5 * lof_h_norm) * 100).round(1)
holdout_recall = (holdout_feat["fraud_risk_score"] >= CHOSEN_THRESHOLD).mean()
print(f"Known-fake holdout set (n={len(holdout_feat)}): "
      f"{holdout_recall:.1%} correctly scored >= high-risk threshold ({CHOSEN_THRESHOLD})")
holdout_feat[["handle", "followers", "follower_growth_30d_pct", "fraud_risk_score"]].sort_values(
    "fraud_risk_score", ascending=False).head(10)

Known-fake holdout set (n=150): 100.0% correctly scored >= high-risk threshold (16)


,handle,followers,follower_growth_30d_pct,fraud_risk_score
0,@knownfake_0000,128753,125.202988,93.2
60,@knownfake_0060,29311,460.657700,82.6
86,@knownfake_0086,49168,420.726254,81.5
102,@knownfake_0102,238326,260.748944,74.5
62,@knownfake_0062,227190,191.644370,74.1
57,@knownfake_0057,75128,303.035383,72.0
65,@knownfake_0065,112907,151.702466,70.4
145,@knownfake_0145,93793,197.334137,67.7
135,@knownfake_0135,81521,372.296737,66.4
139,@knownfake_0139,57784,383.468466,65.0


## 9. Export Deliverables

Final scored CSV (all 5,000 profiles + score + action + validation-sample flag) and the plots used in the
business report.

In [45]:
export_cols = [
    "handle", "platform", "followers", "following", "posts", "avg_likes", "avg_comments",
    "account_age_days", "follower_growth_30d_pct", "engagement_rate", "comment_like_ratio",
    "follower_following_ratio", "pct_comments_generic", "posting_regularity_cv",
    "audience_overlap_score", "iso_forest_score", "lof_score", "fraud_risk_score",
    "vetting_action", "manually_labeled",
]
export_df = feat_df[export_cols].sort_values("fraud_risk_score", ascending=False)
os.makedirs("outputs", exist_ok=True)
export_df.to_csv("outputs/influencer_fraud_scores.csv", index=False)
print(f"Exported {len(export_df)} scored profiles.")
export_df.head(10)

Exported 5000 scored profiles.


,handle,platform,followers,following,posts,avg_likes,avg_comments,account_age_days,follower_growth_30d_pct,engagement_rate,comment_like_ratio,follower_following_ratio,pct_comments_generic,posting_regularity_cv,audience_overlap_score,iso_forest_score,lof_score,fraud_risk_score,vetting_action,manually_labeled
3978,@influencer_03978,Instagram,16686,215,728,2296,385,49.550534,7.284294,0.160674,0.167625,77.609302,0.865560,0.140295,0.724435,0.853100,0.988224,92.1,Reject / Escalate,False
3991,@influencer_03991,Instagram,2833,1527,3727,238,10,249.915665,-1.423252,0.087540,0.040312,1.855272,0.309541,0.294127,0.237220,0.471479,1.000000,73.6,Reject / Escalate,False
4746,@influencer_04746,TikTok,30979,1020,1274,7916,2356,191.925503,4.344126,0.331579,0.297620,30.371569,0.799337,0.066187,0.623865,1.000000,0.405505,70.3,Reject / Escalate,False
3916,@influencer_03916,TikTok,228013,162,51,36690,6797,386.285869,-0.999836,0.190722,0.185261,1407.487654,0.880229,0.150225,0.728867,0.864512,0.439022,65.2,Reject / Escalate,False
823,@influencer_00823,Instagram,6112,4727,1927,79,0,216.894079,99.258621,0.012925,0.004263,1.292998,0.669570,1.282761,0.473668,0.817199,0.472286,64.5,Reject / Escalate,False
2079,@influencer_02079,Instagram,37089,1298,2491,712,30,190.103600,-1.517209,0.020006,0.042375,28.573960,0.257070,0.619028,0.177529,0.439723,0.828917,63.4,Reject / Escalate,False
2330,@influencer_02330,TikTok,388731,43,175,2812,17,1228.817693,98.447864,0.007278,0.005926,9040.255814,0.444957,1.021960,0.801746,0.841734,0.397097,61.9,Reject / Escalate,True
4343,@influencer_04343,Instagram,503711,284,40,16597,337,1000.548409,182.650323,0.033618,0.020319,1773.630282,0.072881,1.428252,0.496400,0.829133,0.375746,60.2,Reject / Escalate,True
4904,@influencer_04904,TikTok,3137,4107,3903,42,1,931.599108,115.381691,0.013707,0.026315,0.763818,0.427850,1.283203,0.762594,0.801908,0.363416,58.3,Reject / Escalate,False
3792,@influencer_03792,Instagram,26902,908,678,212,3,153.948287,163.679769,0.007992,0.015204,29.627753,0.358433,0.527062,0.559189,0.704393,0.449581,57.7,Reject / Escalate,False


## 10. Summary

- **5,000** simulated public profiles scored, unsupervised (Isolation Forest + LOF ensemble), 0-100 Fraud Risk Score.
- Threshold tuned on the 200-profile manually labeled sample to the lowest score that clears **85% precision**
  on the fraud class, then locked before touching the holdout set.
- Validated against a 150-profile "known fake influencer" stress test the models never trained or tuned on.
- 4-tier vetting action policy (Approve / Low-Concern Review / High-Concern Review / Reject-Escalate) exported
  alongside every profile's score for the vetting team to act on directly.
- See the accompanying PDF for threshold rationale, business trade-offs, and rollout recommendations.
print("Notebook run complete.")